# eu_and_v3 — single-switch report: Open, the expected longest open run of the stem

For each of the first `N_DESIGNS` designs of the NucSyn CSV this notebook estimates, by sampling
structures from the folding ensemble,

* **Open (switch + trigger)** = expected length of the **longest run of consecutive** open `stem_up` nt
  for the switch with its real trigger;
* **Open (switch alone)** = the same for the switch without any trigger;
* next to each, **E[k]** = expected number of open nt anywhere on the stem (consecutive or not);

checks the sampler against the exact base-pair-probability matrix, defines
**ON** = Open (switch + trigger) and **OFF** = Open (switch alone) as percentages of the stem, ranks the
designs by **ON - OFF** (percentage points), and also lists the ratio ON / OFF. It writes `var/reports/eu_and_v3_report_<timestamp>.html` (interactive, with a Details button per design) and `.pdf` (the table, then about three pages of details per design; the Details links of the table jump to them, with bookmarks) (plus a CSV
of every design) with the correlation between this ranking and the CSV's own rank.

The definitions are written out in the report itself.

## Setup

In [ ]:
import sys, pathlib

for _base in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents):
    if (_base / "_fixtures.py").exists():
        if str(_base) not in sys.path:
            sys.path.insert(0, str(_base))
        break
for _base in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents):
    if (_base / "_switch_construct.py").exists():
        if str(_base) not in sys.path:
            sys.path.insert(0, str(_base))
        break

import _fixtures as fx
fx.bootstrap()

## Parameters — every tunable in one place

In [ ]:
# ---- data source ------------------------------------------------------------
CSV_PATH = (
    "/Users/zeev/Projects/NucSyn_Public_Platform_0.1.1/nucsyn-platform-work/"
    "output_T3/targets/sele/basic_switch/results_ranked.csv"
)
COLUMNS = {
    "switch": "switch_strand",
    "trigger": "trigger_seq",
    "domains": "switch_domain_breakdown_json",
    "rank": "postgen_global_rank",
    "trigger_start": "trigger_start_pos",
    "trigger_end": "trigger_end_pos",
}
CARRY_METRICS = ("sim_score", "mcc", "switch_toehold_unpaired_prob_mean", "energy_trigger_mfe")
N_DESIGNS = 50            # first rows of the CSV that are scored

# ---- folding and sampling ------------------------------------------------------
TEMPERATURE_C = 37.0
N_SAMPLES = 2000          # structures drawn per state
SEED = 1                  # ViennaRNA's generator is process-global; fixed so a rerun repeats
MAX_WORKERS = 6           # process-pool width
CHECK_SEED_OFFSET = 1000  # a second, independent sampling run (seed + this) only measures the noise

# ---- verification --------------------------------------------------------------
VERIFY_SIGMA = 4.0        # allowed disagreement with the matrix, in standard errors of the sampled mean
VERIFY_FLOOR_NT = 0.02    # plus this many nt (rounding of the matrix, rare draws)

# ---- report --------------------------------------------------------------------
CONTROL_EXTRA_TRIGGERS = ()  # optional extra RNA sequences offered as mismatched controls besides the other designs' triggers
N_CONTROLS = 5            # mismatched triggers per design; OFF is the worst of the alone state and these
CONTROL_MAX_OVERLAP_NT = 8  # the mismatched control trigger shares at most this many nt of identity with the real one
OFF_FLOOR_PCT = 1.0       # OFF (% of the stem) is not allowed below this in the ratio ON / OFF
REPORT_TOP_N = 20         # designs listed in the report table (the CSV export has all)

# Re-render from a saved results file instead of sampling again (None = sample).
RESULTS_TO_LOAD = None

## Load the designs and sample both states

In [ ]:
import collections, datetime, html, json, math, statistics, subprocess, time
from pathlib import Path

from _joint_state_parallel import pick_fake_triggers, use_fold_engine
from _sampled_stem_parallel import DesignJob, run_parallel_designs
from _switch_construct import load_ranked_designs

folder = fx.fold_engine(real=True, temperature=TEMPERATURE_C)
use_fold_engine(folder)   # the pool forks: publish the one engine first

_report_root = Path.cwd()
for _base in (_report_root, *_report_root.parents):
    if (_base / "pyproject.toml").exists():
        _report_root = _base / "var" / "reports"
        break
_report_root.mkdir(parents=True, exist_ok=True)

designs, problems = load_ranked_designs(
    CSV_PATH, folder, columns=COLUMNS, carry_metrics=CARRY_METRICS, limit=N_DESIGNS,
)
for _line_no, _reason in problems[:10]:
    print(f"  skipped line {_line_no}: {_reason}")
print(f"loaded {len(designs)} designs ({len(problems)} rows skipped)")

if RESULTS_TO_LOAD is None:
    # control: for each design a mismatched trigger, i.e. the real trigger of another design that
    # shares at most CONTROL_MAX_OVERLAP_NT nt of identical sequence with the true trigger
    _control_pool = designs + [{"rank": f"extra{_k}", "trigger": _t} for _k, _t in enumerate(CONTROL_EXTRA_TRIGGERS)]
    for _i, _d in enumerate(designs):
        _fakes = pick_fake_triggers((_d["trigger"],), {_d["rank"]}, _control_pool, N_CONTROLS,
                                    start_offset=_i + 1, max_overlap_nt=CONTROL_MAX_OVERLAP_NT)
        _d["control_triggers"] = tuple(_f["trigger"] for _f in _fakes)
        _d["control_ranks"] = [_f["rank"] for _f in _fakes]
    jobs = [
        DesignJob(
            key=_d["rank"], switch=_d["switch"], trigger=_d["trigger"],
            stem_up=_d["domains"]["stem_up"], stem_down=_d["domains"]["stem_down"],
            temperature=TEMPERATURE_C, n_samples=N_SAMPLES, seed=SEED, controls=_d["control_triggers"],
        )
        for _d in designs
    ]
    _t0 = time.time()
    results = run_parallel_designs(jobs, max_workers=MAX_WORKERS)
    check_results = run_parallel_designs(
        [j._replace(seed=SEED + CHECK_SEED_OFFSET) for j in jobs], max_workers=MAX_WORKERS)
    print(f"sampled {len(jobs)} designs twice (second run = noise check) in {time.time() - _t0:.0f}s")
    _results_path = _report_root / f"eu_and_v3_results_{datetime.datetime.now():%Y%m%dT%H%M%S}.json"
    _results_path.write_text(json.dumps({"results": results, "check": check_results}))
    print(f"wrote {_results_path}")
else:
    _saved = json.loads(Path(RESULTS_TO_LOAD).read_text())
    results, check_results = _saved["results"], _saved["check"]
    print(f"loaded results for {len(results)} designs from {RESULTS_TO_LOAD}")

## Open, checks and ranking

Per state: `run_nt_sampled` is the mean over the draws of the longest open run r (this is Open),
`open_nt_sampled` the mean number of open nt k, `open_nt_matrix` the exact mean number of open nt.
The matrix cannot give Open, so a state is *verified* when (1) the sampled mean k matches the matrix
within `VERIFY_SIGMA` standard errors (+ `VERIFY_FLOOR_NT` nt), which shows the sampler is right for that state, and (2) Open
does not exceed the matrix mean k (r <= k always).

ON = 100 x Open(switch + trigger) / L and OFF = 100 x Open(switch alone) / L, both in % of the stem.
Designs are ranked by ON - OFF (percentage points), largest first; ties are broken by the CSV order.
The ratio ON / max(OFF, `OFF_FLOOR_PCT`) is listed for information and orders nothing.

In [ ]:
def _state_ok(state, stem_len):
    run, k, m = state["run_nt_sampled"], state["open_nt_sampled"], state["open_nt_matrix"]
    if run is None or k is None:
        return False
    # standard error of the sampled mean k; sd(k) <= L/2 is used when the sd was not saved
    sd = state.get("open_nt_sd", stem_len / 2.0)
    tol = VERIFY_SIGMA * sd / math.sqrt(state["n_drawn"]) + VERIFY_FLOOR_NT
    return abs(k - m) <= tol and run <= m + tol


scored = []
for _d in designs:
    _r = results[_d["rank"]]
    _L = _d["domains"]["stem_up"][1] - _d["domains"]["stem_up"][0]
    scored.append({
        "design": _d, "csv_rank": int(_d["rank"]), "stem_len": _L,
        "open_t": _r["on"]["run_nt_sampled"], "k_t": _r["on"]["open_nt_sampled"],
        "k_t_matrix": _r["on"]["open_nt_matrix"], "ok_t": _state_ok(_r["on"], _L),
        "k_c_matrix": max((c["open_nt_matrix"] for c in _r.get("controls", [])), default=None),
        "n_controls": len(_r.get("controls", [])),
        "open_a": _r["alone"]["run_nt_sampled"], "k_a": _r["alone"]["open_nt_sampled"],
        "k_a_matrix": _r["alone"]["open_nt_matrix"], "ok_a": _state_ok(_r["alone"], _L),
    })

# ON and OFF as % of the stem; a missing Open makes them (and the difference) None, never 0
for _r in scored:
    _ok = _r["open_t"] is not None and _r["open_a"] is not None
    _r["on_pct"] = 100.0 * _r["open_t"] / _r["stem_len"] if _r["open_t"] is not None else None
    _r["off_alone_pct"] = 100.0 * _r["open_a"] / _r["stem_len"] if _r["open_a"] is not None else None
    # control: exact matrix E[k] with a mismatched trigger is an upper bound of its Open (the sampler is
    # unreliable for mismatched states); the worst (largest) of the controls
    _r["ctrl_pct"] = 100.0 * _r["k_c_matrix"] / _r["stem_len"] if _r["k_c_matrix"] is not None else None
    # OFF = the worst (largest) of the alone state and the controls
    _r["off_pct"] = (max(_r["off_alone_pct"], _r["ctrl_pct"] or 0.0)
                     if _r["off_alone_pct"] is not None else None)
    _r["off_source"] = "control" if (_r["ctrl_pct"] or 0.0) > (_r["off_alone_pct"] or 0.0) else "alone"
    _r["diff_pp"] = _r["on_pct"] - _r["off_pct"] if _ok else None
    _r["ratio"] = _r["on_pct"] / max(_r["off_pct"], OFF_FLOOR_PCT) if _ok else None
    _r["ratio_floored"] = _ok and _r["off_pct"] < OFF_FLOOR_PCT
    # dG: G(switch + trigger) - G(switch alone) - G(trigger alone), ensemble free energies (kcal/mol), exact
    _d = _r["design"]
    _r["dg"] = (folder.partition(f"{_d['switch']}&{_d['trigger']}") - folder.partition(_d["switch"])
                - folder.partition(_d["trigger"]))


# rank by ON - OFF, largest first; a design whose difference is None sorts last
scored.sort(key=lambda r: (r["diff_pp"] is None, -(r["diff_pp"] or 0.0), r["csv_rank"]))
for _pos, _r in enumerate(scored, start=1):
    _r["our_rank"] = _pos

# seed-to-seed noise of ON - OFF: the same designs sampled again with another seed
for _r in scored:
    _c = check_results[_r["design"]["rank"]]
    _on2, _off2 = _c["on"]["run_nt_sampled"], _c["alone"]["run_nt_sampled"]
    _r["diff_pp_check"] = (100.0 * _on2 / _r["stem_len"] - max(100.0 * _off2 / _r["stem_len"], _r["ctrl_pct"] or 0.0)
                          if _on2 is not None and _off2 is not None else None)
_noise = sorted(abs(r["diff_pp"] - r["diff_pp_check"]) for r in scored
                if r["diff_pp"] is not None and r["diff_pp_check"] is not None)
noise_median = statistics.median(_noise)
noise_p95 = _noise[int(0.95 * (len(_noise) - 1))]
n_tied_with_best = sum(1 for r in scored
                       if r["diff_pp"] is not None and scored[0]["diff_pp"] - r["diff_pp"] <= noise_p95)
# near-ties: designs within the noise margin of the best are one tied group, ordered by dG
# (more negative first); the rest follow by ON - OFF
_tied = [r for r in scored if r["diff_pp"] is not None and scored[0]["diff_pp"] - r["diff_pp"] <= noise_p95]
_rest = [r for r in scored if r not in _tied]
_tied.sort(key=lambda r: (r["dg"], r["csv_rank"]))
scored = _tied + _rest
for _pos, _r in enumerate(scored, start=1):
    _r["our_rank"] = _pos
    _r["tied"] = _r in _tied
print(f"seed-to-seed |change of ON - OFF|: median {noise_median:.2f}, 95th percentile {noise_p95:.2f} pp; "
      f"{n_tied_with_best} designs within that of rank 1")

n_ok_t = sum(r["ok_t"] for r in scored)
n_ok_a = sum(r["ok_a"] for r in scored)
print(f"{len(scored)} designs; verified: switch + trigger {n_ok_t}, switch alone {n_ok_a}")
print("top 5 (our rank, CSV rank, ON - OFF):",
      [(r["our_rank"], r["csv_rank"], round(r["diff_pp"], 1)) for r in scored[:5]])

## Correlation with the CSV rank

Spearman rho is the Pearson correlation of the two rank vectors; Kendall tau-b is
(C - D) / sqrt((n0 - n1)(n0 - n2)) over all pairs. Both are +1 when the orders agree.

In [ ]:
def _average_ranks(values):
    order = sorted(range(len(values)), key=lambda i: values[i])
    ranks = [0.0] * len(values)
    i = 0
    while i < len(order):
        j = i
        while j + 1 < len(order) and values[order[j + 1]] == values[order[i]]:
            j += 1
        for k in range(i, j + 1):
            ranks[order[k]] = (i + j) / 2 + 1
        i = j + 1
    return ranks


def spearman_rho(x, y):
    rx, ry = _average_ranks(x), _average_ranks(y)
    mx, my = sum(rx) / len(rx), sum(ry) / len(ry)
    cov = sum((a - mx) * (b - my) for a, b in zip(rx, ry))
    vx, vy = sum((a - mx) ** 2 for a in rx), sum((b - my) ** 2 for b in ry)
    return cov / math.sqrt(vx * vy) if vx and vy else None


def kendall_tau_b(x, y):
    n = len(x)
    concordant = discordant = ties_x = ties_y = 0
    for i in range(n):
        for j in range(i + 1, n):
            dx, dy = x[i] - x[j], y[i] - y[j]
            if dx == 0 and dy == 0:
                continue
            if dx == 0:
                ties_x += 1
            elif dy == 0:
                ties_y += 1
            elif (dx > 0) == (dy > 0):
                concordant += 1
            else:
                discordant += 1
    denom = math.sqrt((concordant + discordant + ties_x) * (concordant + discordant + ties_y))
    return (concordant - discordant) / denom if denom else None


_ranked = [r for r in scored if r["diff_pp"] is not None]
csv_ranks = [r["csv_rank"] for r in _ranked]
our_ranks = [r["our_rank"] for r in _ranked]
rho = spearman_rho(csv_ranks, our_ranks)
tau = kendall_tau_b(csv_ranks, our_ranks)
print(f"Spearman rho = {rho:.3f}   Kendall tau-b = {tau:.3f}   over {len(_ranked)} designs")

In [ ]:
import base64, io
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

plt.rcParams.update({"font.family": "serif", "mathtext.fontset": "cm"})
fig, ax = plt.subplots(figsize=(6.4, 6.0))
ax.scatter(csv_ranks, our_ranks, s=14, c="#7f8c8d", alpha=0.7, label="all designs")
_top = [r for r in _ranked if r["our_rank"] <= REPORT_TOP_N]
ax.scatter([r["csv_rank"] for r in _top], [r["our_rank"] for r in _top], s=22, c="#c0392b",
           label=f"top {REPORT_TOP_N} (listed in the report)")
ax.plot([1, len(_ranked)], [1, len(_ranked)], "k--", lw=0.8, label="same rank in both")
ax.set_xlabel("rank in the CSV (1 = best)")
ax.set_ylabel("rank by ON - OFF, this report (1 = best)")
ax.invert_yaxis()
ax.set_title(f"Spearman rho = {rho:.2f},  Kendall tau-b = {tau:.2f}")
ax.legend(fontsize=8, loc="lower left")
ax.grid(alpha=0.3)
fig.tight_layout()
_buf = io.BytesIO()
fig.savefig(_buf, format="png", dpi=130)
plt.close(fig)
_corr_png_path = _report_root / f"eu_and_v3_correlation_{datetime.datetime.now():%Y%m%dT%H%M%S}.png"
_corr_png_path.write_bytes(_buf.getvalue())
print(f"wrote correlation plot (outside the report) -> {_corr_png_path}")

## Details for each listed design

For each design in the report table: its parts (domains of the switch and the trigger), and, for the switch
alone and for the switch with its trigger, a 2D picture of the minimum-free-energy structure and the
base-pair-probability matrix. The structure is the single most stable one, drawn from `FoldEngine.mfe`;
the matrix is the exact `FoldEngine.base_pair_probabilities`. Opened by the Details button of the HTML report.

In [ ]:
from _report_figures import DOMAIN_COLORS, bpp_heatmap_png, domain_fill, structure_svg


def _fmt_nt(x):
    return "n/a" if x is None else f"{x:.2f}"


_DOMAIN_ORDER = ("prefix", "toehold", "stem_up", "loop", "stem_down", "pre_kozak", "kozak")


def _switch_line_html(design):
    # the whole switch sequence 5' to 3' through the Kozak, each part shaded in its own colour
    kozak_end = design["domains"]["kozak"][1]
    pieces, legend = [], []
    for name in _DOMAIN_ORDER:
        start, end = design["domains"].get(name, (0, 0))
        if end > start:
            colour = DOMAIN_COLORS.get(name, "#C7D0CA")
            pieces.append(f"<span class='dom' style='background:{colour}55' title='{name}'>"
                          f"{design['switch'][start:end]}</span>")
            legend.append(f"<span class='sw' style='background:{colour}'></span>{name}")
    return (f"<p class='note'>Switch sequence, 5&prime; to 3&prime; through the Kozak ({kozak_end} nt):</p>"
            f"<p class='seqline'>{''.join(pieces)}</p><p class='note'>{' &ensp; '.join(legend)}</p>"
            f"<p class='note'>Trigger ({len(design['trigger'])} nt): <span class='seqline'>{design['trigger']}</span></p>")


def _domain_table_html(design):
    switch, trigger = design["switch"], design["trigger"]
    rows = []
    for name in _DOMAIN_ORDER:
        start, end = design["domains"].get(name, (0, 0))
        if end > start:
            rows.append(
                f"<tr><td>{name}</td><td class='n'>{start + 1}&ndash;{end}</td>"
                f"<td class='n'>{end - start}</td><td class='seq'>{switch[start:end]}</td></tr>"
            )
    rows.append(f"<tr><td>trigger</td><td class='n'>&ndash;</td><td class='n'>{len(trigger)}</td>"
                f"<td class='seq'>{trigger}</td></tr>")
    return ("<table class='booktabs small'><tr><th>part</th><th class='n'>positions (1-based)</th>"
            "<th class='n'>length (nt)</th><th>sequence (5&prime; to 3&prime;)</th></tr>"
            + "".join(rows) + "</table>")


def _state_figures(design, strands, heading):
    # (2D structure SVG, base-pair-probability PNG as base64) of one state
    switch = design["switch"]
    sequence = strands.replace("&", "")
    structure = folder.mfe(strands).structure.replace("&", "")
    kozak_end = design["domains"]["kozak"][1] if design["domains"].get("kozak") else len(switch)
    fills = [f for f in domain_fill(design["domains"]) if f[0] < kozak_end]
    svg = structure_svg(
        folder.layout_coordinates(structure), sequence, structure,
        f"{heading} (minimum-free-energy structure)", domain_fill=fills,
        visible_len=kozak_end, construct_len=len(switch),
    )
    boundary = len(switch) if "&" in strands else None
    png = bpp_heatmap_png(sequence, folder.base_pair_probabilities(strands),
                          f"{heading}: base-pair probabilities", boundary)
    return svg, png


for _r in scored[:REPORT_TOP_N]:
    _d = _r["design"]
    _L = _r["stem_len"]
    _on_text = (f"Open = {_fmt_nt(_r['open_t'])} nt; E[k] = {_fmt_nt(_r['k_t'])} nt "
                f"(matrix {_fmt_nt(_r['k_t_matrix'])}); L = {_L}")
    _ctrl_text = (f"worst E[k] from the matrix over {_r['n_controls']} mismatched triggers = {_fmt_nt(_r['k_c_matrix'])} nt "
                  f"({_r['ctrl_pct']:.1f}% of L; an upper bound of Open); OFF = larger of this and Open(alone) = {_r['off_pct']:.1f}% ({_r['off_source']}); "
                  f"mismatched triggers = triggers of CSV rows {', '.join(str(x) for x in _d.get('control_ranks', []))}; L = {_L}")
    _off_text = (f"Open = {_fmt_nt(_r['open_a'])} nt; E[k] = {_fmt_nt(_r['k_a'])} nt "
                 f"(matrix {_fmt_nt(_r['k_a_matrix'])}); L = {_L}")
    _svg_off, _png_off = _state_figures(_d, _d["switch"], "OFF: switch alone")
    _svg_on, _png_on = _state_figures(_d, _d["switch"] + "&" + _d["trigger"], "ON: switch + trigger")
    _r["detail_html"] = (
        f"<div class='panel' id='p{_r['our_rank']}'><h3>Design rank {_r['our_rank']} "
        f"&middot; stem L = {_L} nt "
        f"<a class='back' href='#results'>&uarr; back to the table</a></h3>"
        f"{_switch_line_html(_d)}{_domain_table_html(_d)}"
        f"<div class='state'><h4>OFF: switch alone</h4><p class='note'>{_off_text}</p>{_svg_off}</div>"
        f"<div class='state'><h4>ON: switch + trigger</h4><p class='note'>{_on_text}</p>{_svg_on}</div>"
        f"<p class='note'><b>Controls: switch + mismatched triggers.</b> {_ctrl_text}</p>"
        f"<div class='bppRow'>"
        f"<figure><img src='data:image/png;base64,{_png_off}'>"
        f"<figcaption>OFF: switch alone</figcaption></figure>"
        f"<figure><img src='data:image/png;base64,{_png_on}'>"
        f"<figcaption>ON: switch + trigger</figcaption></figure>"
        f"</div></div>"
    )
print(f"details built for {REPORT_TOP_N} designs")

## Report

In [ ]:
def _fmt_nt(x):
    return "n/a" if x is None else f"{x:.2f}"


def _fmt_nt_pct(x, length):
    return "n/a" if x is None else f"{x:.2f}&nbsp;({100.0 * x / length:.0f}%)"


_CSS = r'''
@import url("https://cdn.jsdelivr.net/gh/aaaakshat/cm-web-fonts@latest/fonts.css");
@page { size: A4; margin: 20mm 18mm; }
body { font-family: "Computer Modern Serif", "CMU Serif", "Latin Modern Roman", "Times New Roman", serif;
       font-size: 11.5pt; line-height: 1.5; color: #111; max-width: 174mm; margin: 0 auto; }
h1 { font-size: 20pt; font-weight: normal; text-align: center; margin: 0 0 4pt 0; line-height: 1.25; }
.byline { text-align: center; font-size: 10pt; color: #444; margin-bottom: 18pt; }
h2 { font-size: 13.5pt; margin: 22pt 0 6pt 0; font-weight: bold; break-after: avoid; }
p { margin: 6pt 0; text-align: justify; }
dl { margin: 6pt 0; } dt { break-after: avoid; font-weight: bold; float: left; clear: left; width: 34mm; }
dd { break-inside: avoid; margin: 0 0 4pt 36mm; }
code { font-family: "Courier New", monospace; font-size: 9.5pt; }
table { border-collapse: collapse; margin: 8pt auto; font-size: 6.8pt; }
table.booktabs { border-top: 1.5pt solid #111; border-bottom: 1.5pt solid #111; }
table.booktabs th { border-bottom: 0.75pt solid #111; font-weight: bold; padding: 3pt 2pt; text-align: left; }
table.booktabs th.n { white-space: nowrap; }
table.booktabs td { padding: 2pt 2pt; }
td.n, th.n { text-align: right; font-variant-numeric: tabular-nums; }
.ok { color: #1e7e34; } .no { color: #b02a37; font-weight: bold; }
figure { margin: 10pt 0; text-align: center; break-inside: avoid; }
figcaption { font-size: 10pt; color: #333; margin-top: 4pt; text-align: left; }
.note { font-size: 10pt; color: #333; }
.katex { font-size: 1.0em !important; } .katex-display { margin: 0.55em 0 !important; }
th.grp { text-align: center; border-bottom: 0.75pt solid #111; }
th.on { background: #fbe9e6; } th.off { background: #e6eef7; } th.ctl { background: #eef3e6; }
td.okk { background: #e3f3e6; } td.badk { background: #fbe3e0; }
tr.detail > td { padding: 8pt 4pt 14pt 4pt; border-bottom: 0.75pt solid #999; }
.panel h3 { font-size: 12pt; margin: 0 0 6pt 0; } .state h4 { font-size: 11pt; margin: 6pt 0 2pt 0; }
.state { margin-top: 10pt; }
.bppRow { display: flex; gap: 12pt; justify-content: center; margin-top: 10pt; break-inside: avoid; }
.bppRow figure { flex: 1; margin: 2pt; text-align: center; } .bppRow img { max-width: 100%; }
td.seq { font-family: "Courier New", monospace; font-size: 8.5pt; word-break: break-all; }
table.small { font-size: 8.5pt; margin: 4pt 0; }
@media print { .detail, .detail-btn, .noprint { display: none !important; } }
.detail-link, .back, .detail-btn { white-space: nowrap; font-family: inherit; font-size: 9.5pt; color: #1a5fb4;
  text-decoration: underline; font-weight: normal; border: none; background: none; padding: 0; cursor: pointer; }
.back { margin-left: 14pt; }
.seqline { font-family: "Courier New", monospace; font-size: 9pt; word-break: break-all; line-height: 1.7; }
.dom { padding: 1pt 0; } .sw { display: inline-block; width: 8pt; height: 8pt; margin-right: 3pt; }
.abstract { border-left: 3pt solid #1F6FEB; padding: 2pt 8pt; margin: 8pt 0 12pt 0; font-size: 11pt; background: #f4f7fb; }
.appendix .panel { break-before: page; } .appendix .panel:first-of-type { break-before: auto; } h2 + table { break-inside: avoid; } table.small, table.small tr { break-inside: avoid; } #results + p { break-after: avoid; break-inside: avoid; } .appendix .state { break-inside: avoid; }
.appendix svg { max-height: 112mm !important; }
'''

_TEMPLATE = r'''<!doctype html><html><head><meta charset="utf-8"><title>eu_and_v3 report</title>
<link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/katex@0.16.9/dist/katex.min.css">
<script defer src="https://cdn.jsdelivr.net/npm/katex@0.16.9/dist/katex.min.js"></script>
<script defer src="https://cdn.jsdelivr.net/npm/katex@0.16.9/dist/contrib/auto-render.min.js"></script>
<script>document.addEventListener("DOMContentLoaded", function () {
  renderMathInElement(document.body, {delimiters: [
    {left: "\\[", right: "\\]", display: true}, {left: "\\(", right: "\\)", display: false}]});
});</script>
<script>function toggleDetail(id){var r=document.getElementById(id);r.hidden=!r.hidden;}</script>
<style>@@CSS@@</style></head><body>

<h1>Single-switch screen</h1>
<div class="byline">@@SOURCE@@<br>First @@N_DESIGNS@@ designs (the table lists the best @@REPORT_TOP_N@@), folded at
@@TEMPERATURE@@&nbsp;&deg;C with ViennaRNA @@VRNA@@ &middot; @@DATE@@</div>

<div class="abstract"><b>Summary.</b> @@ABSTRACT@@</div>

<h2>1&ensp;Definitions</h2>
<dl>
<dt>switch</dt><dd>the RNA switch strand of a design. It contains one hairpin.</dd>
<dt>trigger</dt><dd>the design's real trigger RNA.</dd>
<dt>stem</dt><dd>the hairpin stem has two sides of \(L\) nucleotides each, \(\mathrm{stem}_{\uparrow}\) (5&prime; side)
and \(\mathrm{stem}_{\downarrow}\) (3&prime; side). In the closed hairpin they pair with each other.
Here \(L\) = @@STEM_LENS@@.</dd>
<dt>state</dt><dd>a set of strands folded together: <i>switch + trigger</i> (the switch with its real trigger),
<i>switch alone</i> (no trigger) or a <i>control</i> (the switch with a mismatched trigger).</dd>
<dt>mismatched trigger</dt><dd>the real trigger of another design of the same table (or, if none qualifies, another sequence offered for this purpose) that shares at most @@CONTROL_OVERLAP@@ nt of identical sequence with the true trigger; @@N_CONTROLS@@ of them per design are used as wrong inputs.</dd>
<dt>prefix, Kozak, payload</dt><dd>the switch RNA starts with a short <i>prefix</i>, then the toehold and the hairpin; the <i>Kozak</i> sequence follows the
hairpin and the <i>payload</i> (the coding sequence, 67 nt here) follows the Kozak. The payload is folded together with the rest but is not drawn.</dd>
<dt>how to read the figures</dt><dd>2D structures: every circle is a nucleotide coloured by part; a <b>gold ring</b> marks a nucleotide paired into the payload
(the undrawn &ldquo;hidden tail&rdquo; after the Kozak) and a <b>pink ring</b> one paired to the (undrawn) trigger strand. Heat maps: the axes run over the
whole folded RNA, including the payload and, for the switch + trigger state, the trigger; the dotted red lines mark where one strand ends and the next begins.</dd>
<dt>toehold, hairpin, loop</dt><dd>the <i>toehold</i> is the single-stranded stretch of the switch where the trigger first binds; the <i>hairpin</i> is the stem (\(\mathrm{stem}_{\uparrow}\) paired with \(\mathrm{stem}_{\downarrow}\)) closed by a <i>loop</i> of unpaired nucleotides. Binding the trigger is meant to open the hairpin.</dd>
<dt>pp</dt><dd>percentage points, the unit of a difference of two percentages.</dd>
<dt>structure</dt><dd>one folded arrangement \(s\) of the strands, i.e. a set of base pairs.</dd>
<dt>\(E(s)\), \(G\)</dt><dd>the energy of structure \(s\), and the free energy of the whole ensemble of
structures of the state (both in kcal/mol).</dd>
<dt>\(\Delta G\)</dt><dd>\(G_{\text{switch + trigger}}-G_{\text{switch alone}}-G_{\text{trigger alone}}\), in kcal/mol, from the exact partition
function: the free energy gained by forming the complex from the two separate molecules. More negative means the trigger binds the switch more strongly (no concentration term). It is listed for information and orders only the tied group; a more stable hairpin makes it less negative, so the ordering slightly favours less stable switches. Strand concentration is not included (the duplex-initiation term of ViennaRNA's dimer energy is).</dd>
<dt>\(P(s)\)</dt><dd>the probability of \(s\) in the ensemble, \(P(s)=\exp\!\big(-(E(s)-G)/RT\big)\), with \(R\) the gas constant and \(T\) the absolute temperature.</dd>
<dt>open nucleotide</dt><dd>a \(\mathrm{stem}_{\uparrow}\) nucleotide that is <i>not</i> paired to a
\(\mathrm{stem}_{\downarrow}\) nucleotide in \(s\) (any one, including a shifted partner). It may be unpaired, or paired to the trigger or to anything else.</dd>
<dt>\(k(s)\)</dt><dd>the number of open nucleotides in \(s\), from \(0\) to \(L\), anywhere on the stem.</dd>
<dt>\(r(s)\)</dt><dd>the length of the longest stretch of <i>consecutive</i> open nucleotides in \(s\),
from \(0\) to \(L\). Always \(r(s)\le k(s)\). Nucleotides in one stretch are open at the same time.</dd>
</dl>

<h2>2&ensp;Open</h2>
<p>For a state, <b>Open</b> is the expected length of the <b>longest run of consecutive open nucleotides</b>
(not the total number of open nucleotides),
\[ \mathrm{Open}\;=\;\mathbb{E}[r]\;=\;\sum_{s}P(s)\,r(s), \]
reported in nucleotides and as a percentage \(100\cdot\mathrm{Open}/L\) of the stem. It is computed for two states:
\[ \begin{aligned}
\mathrm{Open}_{\text{trigger}}&=\mathbb{E}[r]\ \text{of the switch with its real trigger},\\
\mathrm{Open}_{\text{alone}}&=\mathbb{E}[r]\ \text{of the switch alone}.
\end{aligned} \]
A good switch has \(\mathrm{Open}_{\text{trigger}}\) large (the hairpin opens when its trigger is present) and
\(\mathrm{Open}_{\text{alone}}\) small (it stays closed without it).
</p>
<p>Both are expressed as a percentage of the stem and compared,
\[ \begin{aligned}
&\mathrm{ON}=100\,\mathrm{Open}_{\text{trigger}}/L,\qquad \mathrm{OFF}=100\,\max\big(\mathrm{Open}_{\text{alone}},\,\mathrm{Control}\big)/L,\\[1pt]
&\mathrm{ON}-\mathrm{OFF}\ \text{(percentage points)},\qquad \mathrm{ratio}=\mathrm{ON}\,/\max(\mathrm{OFF},\,@@FLOOR@@).
\end{aligned} \]
OFF is the <b>worst</b> (largest) of the switch alone and the controls (below). <b>The designs are ranked by \(\mathrm{ON}-\mathrm{OFF}\)</b> in percentage points, largest first, with
near-ties handled as described below. A percentage of the stem makes designs with different stem lengths comparable (a count of
nucleotides would favour long stems for ON. For OFF the effect may reverse (fraying at the stem ends is roughly constant in nucleotides), so stem lengths are not fully neutral. Counts would favour long stems, which have more nucleotides that can open). The ratio is listed for
information and orders nothing; the floor of @@FLOOR@@% on OFF keeps it finite when OFF is close to zero, and a
ratio computed with the floor is marked with an asterisk.</p>
<p><b>Controls.</b> To see whether Open responds only to the <i>right</i> trigger, every design is also folded with @@N_CONTROLS@@ mismatched triggers. For each, the exact \(\mathbb{E}[k]_{\text{matrix}}\) of the switch with that trigger is an <b>upper bound</b> on its Open (\(\mathrm{Open}\le\mathbb{E}[k]\)); the matrix is used instead of sampling because the sampler is unreliable for such mismatched states. The <i>Control</i> is the largest of these bounds, \(100\cdot\max\mathbb{E}[k]_{\text{matrix}}/L\), and OFF is the larger of it and the alone value, so a switch that any mismatched trigger opens is penalised. The Control column shows it; ON &minus; OFF therefore already contains it.</p>
<p><b>Ties.</b> Repeating the sampling with another seed changes ON &minus; OFF by a small amount (section 5). Designs whose ON &minus; OFF lies within the
95th-percentile of that change from the best one cannot be told apart from it; they form one <i>tied group</i> (marked &dagger; in the table), ranked inside the group by \(\Delta G\) (more negative first, i.e. stronger binding of the trigger) and
listed before the other designs. Only Monte-Carlo noise is counted here; the error of the energy model is larger, so designs just outside the group are not clearly worse either, and tied designs may share a trigger or toehold, so they are not independent candidates. The tied group is the report's shortlist; &dagger; marks it.</p>
<p><b>What Open does and does not measure.</b> The ON state is the ensemble of the switch <i>with its trigger already bound as a complex</i>; the trigger concentration, the binding kinetics and the fraction of switches that actually form the complex are not modelled. Open measures how far the hairpin stem is opened at equilibrium (the exact ensemble of ViennaRNA at the stated
temperature). It is a structural proxy for the hairpin releasing the region behind it; it is not a measurement of translation, of ribosome binding or
start-codon accessibility, of kinetics, or of behaviour in a cell, and trigger accessibility and self-structure, cross-talk between triggers, the openness of \(\mathrm{stem}_{\downarrow}\) and of the start codon, temperature or parameter-set sensitivity, and the origin of the CSV are not assessed; the only control is one mismatched trigger per design (no scrambled triggers or known reference switch), so the scale of the numbers is not calibrated against experiment.</p>
<p>Next to each Open the report lists \(\mathbb{E}[k]\), the <b>expected number of open nucleotides anywhere on
the stem</b>, whether or not they are consecutive,
\[ \mathbb{E}[k]\;=\;\sum_{s}P(s)\,k(s)\;\ge\;\mathrm{Open}. \]
For the listed designs the two nearly coincide (the open nucleotides are mostly contiguous). It is the more relevant number for a state that should stay closed: a stem that is open in scattered places has a small
Open but may have a larger \(\mathbb{E}[k]\). It is given sampled and from the exact matrix. The stem length
\(L\) of each design is listed too, because Open and \(\mathbb{E}[k]\) are counts out of \(L\).</p>

<h2>3&ensp;Estimation and verification</h2>
<p><i>In plain words:</i> Open is estimated by drawing random structures with the probability they have at equilibrium. The exact base-pair probability matrix cannot give Open but can give the average number of open nucleotides, so it is used to check that the drawn structures behave correctly.</p>
<p><b>Sampled estimate.</b> The sum over all structures cannot be written out, so \(N\) structures
\(S_1,\dots,S_N\) are drawn from the ensemble (ViennaRNA stochastic backtracking,
<code>FoldEngine.sample_structures</code>) and averaged,
\[ \mathbb{E}[r]\;\approx\;\frac1N\sum_{i=1}^{N}r(S_i),\qquad N=@@N_SAMPLES@@ . \]
A structure is drawn with probability \(P(s)\), so this estimates the full sum, including structures that are
never drawn. <b>No normalisation is needed:</b> the draws are already distributed according to \(P(s)\), and the average
needs no division by the share of the ensemble that the distinct sampled structures cover (that share is small, since
almost every draw is a different structure; dividing by it would only be needed if the exact \(P(s)\) were summed over
the distinct sampled structures instead). Its statistical error is \(\mathrm{sd}(r)/\sqrt N\).</p>
<p><b>Matrix check.</b> The base-pair probability matrix \(p_{ij}\) (the probability that nucleotides \(i\) and \(j\)
are paired, summed over all structures) gives the expected <i>number</i> of open nucleotides exactly,
\[ \mathbb{E}[k]\;=\;\sum_{i\in \mathrm{stem}_{\uparrow}}\Big(1-\sum_{j\in \mathrm{stem}_{\downarrow}}p_{ij}\Big). \]
It cannot give \(\mathbb{E}[r]\), because it does not know which positions are open together. It does give
the bound \(\mathbb{E}[r]\le\mathbb{E}[k]\). A state is therefore <i>verified</i> (its \(\mathbb{E}[k]\) cells are shaded light green in the table) when
(1) the sampled \(\mathbb{E}[k]\) agrees with the matrix value to within \(\tau\), which shows the sampler is
right for that state, and (2) \(\mathrm{Open}\le\mathbb{E}[k]_{\text{matrix}}+\tau\) (a sanity bound: \(r\le k\) holds for every draw, so this only catches a broken estimate), with \(\tau\) given below. Green therefore shows that the sampler
reproduces the per-nucleotide openness of that state to within \(\tau\); it does not prove the longest runs themselves, which the matrix cannot see.</p>
<p><b>The tolerance.</b> The sampled mean of \(k\) has a standard error \(\mathrm{sd}(k)/\sqrt N\), where \(\mathrm{sd}(k)\) is the standard deviation of \(k\) over the \(N\) draws of that very state. The tolerance is therefore set per state,
\(\tau=@@VERIFY_SIGMA@@\,\mathrm{sd}(k)/\sqrt N+@@VERIFY_FLOOR@@\) nucleotides, so a state that should stay closed (small \(\mathbb{E}[k]\), small \(\mathrm{sd}\)) is checked tightly and a variable one is checked loosely. The constant @@VERIFY_FLOOR@@ nt only
allows for the rounding of the matrix. Verification catches gross sampler faults; \(\tau\) can be as large as the seed-to-seed noise of the ranking, so a green cell does not guarantee ranking-level accuracy. The tolerance sets the shading only, not any Open or \(\mathbb{E}[k]\) value.</p>
<p><b>Strand order.</b> For the switch + trigger state the trigger strand is written first in the sampler (the order that matched the matrix); the switch alone is a single strand.</p>
<p><b>Unverified states.</b> States that fail the check stay in all medians, noise figures and rank correlations; they are listed in section 5. The sampler's dependence on strand order is empirical, not explained.</p>
<p><b>Limitation.</b> ViennaRNA's sampler was found to return wrong draws for some multi-strand states, depending on which strand is written first
(structures whose energy is far above the ensemble's, always with the stem open, so Open and \(\mathbb{E}[k]\) are biased upwards). The strand order is chosen to avoid this, and no draw is filtered or
corrected. In this run @@N_OK_T@@ of @@N_SCORED@@ switch + trigger states and @@N_OK_A@@ of @@N_SCORED@@ switch-alone states passed the check; a state shaded light red should not be trusted. @@RED_TEXT@@</p>

<h2>4&ensp;Parameters</h2>
<table class="booktabs"><tr><th>name</th><th>value</th><th>meaning</th></tr>
@@PARAMS@@
</table>

<h2>5&ensp;Summary</h2>
<table class="booktabs">
<tr><td>designs scored</td><td class="n">@@N_SCORED@@</td></tr>
<tr><td>switch + trigger verified against the matrix</td><td class="n">@@N_OK_T@@ (@@PCT_OK_T@@%)</td></tr>
<tr><td>switch alone verified against the matrix</td><td class="n">@@N_OK_A@@ (@@PCT_OK_A@@%)</td></tr>
<tr><td>median ON / OFF (% of the stem)</td><td class="n">@@MED_T@@% / @@MED_A@@%</td></tr>
<tr><td>median ON &minus; OFF (percentage points)</td><td class="n">@@MED_D@@</td></tr>
<tr><td>median control (worst of @@N_CONTROLS@@ mismatched triggers, matrix upper bound), % of the stem</td><td class="n">@@MED_C@@%</td></tr>
<tr><td>seed-to-seed change of ON &minus; OFF, second independent run: median / 95th percentile (pp)</td><td class="n">@@NOISE_MED@@ / @@NOISE_P95@@</td></tr>
</table>
<p>@@BEST_TEXT@@</p>

<h2 id="results">6&ensp;First @@REPORT_TOP_N@@ designs (tied group first, then by \(\mathrm{ON}-\mathrm{OFF}\))</h2>
<p class="note">&dagger; = member of the tied group (section 2); inside it the order is by \(\Delta G\), so rank 1 need not have the highest ON &minus; OFF. Green or red shading only says whether the sampler agrees with the matrix; it is not a quality flag for the design. * after a ratio = OFF was below the floor, so the floor was used. \(L\) is the stem length of the design. Open is the expected <b>longest run</b> of consecutive open
nucleotides, in nucleotides with the percentage of \(L\) in brackets. \(\mathbb{E}[k]\) is the expected number of
open nucleotides anywhere on the stem (not necessarily consecutive), sampled and exact (matrix); both are shaded
light green when the sampled \(\mathbb{E}[k]\) agrees with the matrix and Open does not exceed it (section 3), light red
otherwise. Control = the largest exact \(\mathbb{E}[k]\) of the switch with any of @@N_CONTROLS@@ mismatched triggers, an upper bound of its Open. OFF is the larger of the alone Open and the Control (in %). The last columns are ON &minus; OFF (percentage points of the stem), the ratio ON / OFF and \(\Delta G\) (kcal/mol). In the HTML version the Details button of each row opens the design's parts, a 2D structure and the
base-pair-probability matrix for both states.</p>
<table class="booktabs">
<thead><tr><th rowspan="2" class="n">rank</th><th rowspan="2" class="n">\(L\)</th>
<th colspan="3" class="grp on">ON &ensp;<small>switch + trigger</small></th>
<th colspan="3" class="grp off">OFF &ensp;<small>switch alone</small></th>
<th class="grp ctl">Control &ensp;<small>5 mismatched triggers</small></th>
<th rowspan="2" class="n">ON &minus; OFF<br>(pp)</th><th rowspan="2" class="n">ratio<br>ON / OFF</th><th rowspan="2" class="n">\(\Delta G\)<br>(kcal/mol)<br>lower = tighter</th>@@BTNHEAD@@</tr>
<tr><th class="n">\(\mathrm{Open}_{\text{trigger}}\)<br>longest run</th><th class="n">\(\mathbb{E}[k]\)<br>sampled</th>
<th class="n">\(\mathbb{E}[k]\)<br>matrix</th>
<th class="n">\(\mathrm{Open}_{\text{alone}}\)<br>longest run</th><th class="n">\(\mathbb{E}[k]\)<br>sampled</th>
<th class="n">\(\mathbb{E}[k]\)<br>matrix</th>
<th class="n">\(\mathbb{E}[k]\)<br>matrix (&ge; Open)</th></tr></thead>
@@ROWS@@
</table>
@@APPENDIX@@
</body></html>'''

_stem_text = ", ".join(str(n) for n in sorted({r["stem_len"] for r in scored}))
_param_rows = [
    ("N_DESIGNS", N_DESIGNS, "first rows of the CSV that are scored"),
    ("TEMPERATURE_C", TEMPERATURE_C, "folding temperature, &deg;C"),
    ("N_SAMPLES", N_SAMPLES, "structures drawn from the ensemble for each state"),
    ("SEED", SEED, "random seed for the sampler (a rerun repeats)"),
    ("CHECK_SEED_OFFSET", CHECK_SEED_OFFSET, "a second sampling run uses SEED + this; it only measures the noise (section 5)"),
    ("VERIFY_SIGMA", VERIFY_SIGMA, "&tau;: allowed disagreement with the matrix, in standard errors of the sampled mean"),
    ("VERIFY_FLOOR_NT", VERIFY_FLOOR_NT, "added to &tau; (nt)"),
    ("N_CONTROLS", N_CONTROLS, "mismatched triggers per design; OFF is the worst of the alone state and these"),
    ("CONTROL_MAX_OVERLAP_NT", CONTROL_MAX_OVERLAP_NT, "the mismatched control trigger shares at most this many nt of identical sequence with the true trigger"),
    ("OFF_FLOOR_PCT", OFF_FLOOR_PCT, "OFF is not allowed below this many % of the stem in the ratio ON / OFF"),
    ("REPORT_TOP_N", REPORT_TOP_N, "designs listed in the table of section 6"),
]
_params_html = "".join(
    f"<tr><td><code>{n}</code></td><td>{v}</td><td>{m}</td></tr>" for n, v, m in _param_rows
)
def _fmt_pp(x):
    return "n/a" if x is None else f"{x:.1f}"


def _fmt_ratio(r):
    if r["ratio"] is None:
        return "n/a"
    return f"{r['ratio']:.0f}" + ("*" if r["ratio_floored"] else "")


def _row_html(r, interactive=True):
    L = r["stem_len"]
    cls_t = "okk" if r["ok_t"] else "badk"
    cls_a = "okk" if r["ok_a"] else "badk"
    rid = f"d{r['our_rank']}"
    button = (f"<td class='noprint'><button class='detail-btn' onclick=\"toggleDetail('{rid}')\">"
              "Details</button></td>") if interactive else (
        f"<td><a class='detail-link' href='#p{r['our_rank']}'>Details</a></td>")
    detail_row = (f"<tr class='detail' id='{rid}' hidden><td colspan='13'>{r['detail_html']}</td></tr>"
                  if interactive else "")
    return (
        f"<tr><td class='n'>{r['our_rank']}{'&dagger;' if r['tied'] else ''}</td><td class='n'>{L}</td>"
        f"<td class='n'>{_fmt_nt_pct(r['open_t'], L)}</td>"
        f"<td class='n {cls_t}'>{_fmt_nt(r['k_t'])}</td><td class='n {cls_t}'>{_fmt_nt(r['k_t_matrix'])}</td>"
        f"<td class='n'>{_fmt_nt_pct(r['open_a'], L)}</td>"
        f"<td class='n {cls_a}'>{_fmt_nt(r['k_a'])}</td><td class='n {cls_a}'>{_fmt_nt(r['k_a_matrix'])}</td>"
        f"<td class='n'>{_fmt_nt_pct(r['k_c_matrix'], L)}</td>"
        f"<td class='n'><b>{_fmt_pp(r['diff_pp'])}</b></td><td class='n'>{_fmt_ratio(r)}</td>"
        f"<td class='n'>{r['dg']:.1f}</td>"
        f"{button}</tr>{detail_row}"
    )


_rows_html = "".join(_row_html(r) for r in scored[:REPORT_TOP_N])
_n = len(scored)
_med = lambda key: statistics.median(  # noqa: E731
    100.0 * r[key] / r["stem_len"] for r in scored if r[key] is not None)
_fields = {
    "CSS": _CSS, "SOURCE": html.escape(Path(CSV_PATH).name),
    "ABSTRACT": (f"Each of the first {_n} designs of the CSV (in the order of the input file, whose own ranking is not described here) is folded with and without its trigger. <b>ON</b> is how far the hairpin stem is open with the trigger, <b>OFF</b> how far without it or with a wrong trigger (the worst of the alone state and 5 mismatched triggers), "
        f"both as a percentage of the stem (100% = fully open). A good switch has ON near 100% and OFF near 0%. Here the median ON is {statistics.median(r['on_pct'] for r in scored if r['on_pct'] is not None):.1f}% and the median OFF "
        f"{statistics.median(r['off_pct'] for r in scored if r['off_pct'] is not None):.1f}%. Designs are ranked by ON &minus; OFF; {n_tied_with_best} of {_n} lie within the run-to-run difference (95th percentile of the change of ON &minus; OFF between two independent samplings: {noise_p95:.2f} points) of the best and cannot be told apart by that score. "
        f"They are marked &dagger; and are the shortlist; among them pick by \\(\\Delta G\\) (the order of the table) or by the details. This is a structure-based prediction only, with no experimental calibration."), "N_DESIGNS": _n, "TEMPERATURE": f"{TEMPERATURE_C:g}",
    "DATE": datetime.datetime.now().strftime("%Y-%m-%d %H:%M"), "STEM_LENS": _stem_text,
    "N_SAMPLES": N_SAMPLES, "VERIFY_SIGMA": f"{VERIFY_SIGMA:g}", "VERIFY_FLOOR": f"{VERIFY_FLOOR_NT:g}",
    "RED_TEXT": ("All listed designs are shaded green." if all(r["ok_t"] and r["ok_a"] for r in scored[:REPORT_TOP_N])
                 else f"{sum(not (r['ok_t'] and r['ok_a']) for r in scored[:REPORT_TOP_N])} of the listed designs have a red cell."), "PARAMS": _params_html, "N_SCORED": _n,
    "N_OK_T": n_ok_t, "PCT_OK_T": f"{100.0 * n_ok_t / _n:.1f}",
    "N_OK_A": n_ok_a, "PCT_OK_A": f"{100.0 * n_ok_a / _n:.1f}",
    "MED_T": f"{statistics.median(r['on_pct'] for r in scored if r['on_pct'] is not None):.1f}",
    "MED_A": f"{statistics.median(r['off_pct'] for r in scored if r['off_pct'] is not None):.1f}",
    "MED_D": f"{statistics.median(r['diff_pp'] for r in scored if r['diff_pp'] is not None):.1f}",
    "FLOOR": f"{OFF_FLOOR_PCT:g}", "CONTROL_OVERLAP": CONTROL_MAX_OVERLAP_NT, "N_CONTROLS": N_CONTROLS,
    "MED_C": f"{statistics.median(r['ctrl_pct'] for r in scored if r['ctrl_pct'] is not None):.1f}",
    "VRNA": folder.versions().get("ViennaRNA", ""),
    "NOISE_MED": f"{noise_median:.2f}", "NOISE_P95": f"{noise_p95:.2f}",
    "BEST_TEXT": (f"Highest ON &minus; OFF: {max(r['diff_pp'] for r in scored if r['diff_pp'] is not None):.1f} points. "
                  f"{n_tied_with_best} of the {len(scored)} designs lie within the 95th-percentile seed-to-seed change "
                  f"({noise_p95:.2f} points) of it. They form the tied group (&dagger;), which is the shortlist; inside it designs are ordered by \\(\\Delta G\\), not by ON &minus; OFF."),
    "REPORT_TOP_N": REPORT_TOP_N, "ROWS": _rows_html,
}
def _fill(template, fields):
    for _key, _value in fields.items():
        template = template.replace(f"@@{_key}@@", str(_value))
    return template


_btn_head = '<th rowspan="2" class="noprint"></th>'
_appendix = ("<div class='appendix'><h2>7&ensp;Design details</h2><p class='note'>About two pages per design of the "
             "table above (the 2D structures of the switch alone and of the switch with its trigger, then their base-pair probabilities side by side): its parts, and the minimum-free-energy structure and base-pair probabilities of "
             "each state. (The HTML version shows the same under each row's Details button.)</p>"
             + "".join(r["detail_html"] for r in scored[:REPORT_TOP_N]) + "</div>")
# interactive HTML: buttons, details inline. Print version (PDF source): no buttons, details as an appendix.
report_html = _fill(_TEMPLATE, {**_fields, "BTNHEAD": _btn_head, "APPENDIX": ""})
_print_fields = {**_fields, "BTNHEAD": '<th rowspan="2"></th>', "APPENDIX": _appendix,
                 "ROWS": "".join(_row_html(r, interactive=False) for r in scored[:REPORT_TOP_N])}
report_print_html = _fill(_TEMPLATE, _print_fields)

_ts = datetime.datetime.now().strftime("%Y%m%dT%H%M%S")
report_html_path = _report_root / f"eu_and_v3_report_{_ts}.html"
report_html_path.write_text(report_html)
_print_html_path = _report_root / f"eu_and_v3_report_{_ts}_print.html"
_print_html_path.write_text(report_print_html)

import csv as _csv
all_csv_path = _report_root / f"eu_and_v3_all_designs_{_ts}.csv"
with all_csv_path.open("w", newline="") as _fh:
    _w = _csv.writer(_fh)
    _w.writerow(["our_rank", "csv_rank", "stem_len", "Open_trigger_nt", "Ek_trigger_sampled_nt",
                 "Ek_trigger_matrix_nt", "trigger_verified", "Open_alone_nt", "Ek_alone_sampled_nt",
                 "Ek_alone_matrix_nt", "alone_verified", "ON_pct", "OFF_pct", "ON_minus_OFF_pp",
                 "ratio_ON_over_OFF", "dG_kcal_per_mol", "tied_group", "OFF_alone_pct",
                 "control_matrix_upper_bound_pct", "OFF_source", "control_trigger_csv_ranks"])
    for r in scored:
        _w.writerow([r["our_rank"], r["csv_rank"], r["stem_len"], r["open_t"], r["k_t"],
                     r["k_t_matrix"], r["ok_t"], r["open_a"], r["k_a"], r["k_a_matrix"], r["ok_a"],
                     r["on_pct"], r["off_pct"], r["diff_pp"], r["ratio"], r["dg"], r["tied"], r["off_alone_pct"], r["ctrl_pct"],
                     r["off_source"], " ".join(str(x) for x in r["design"].get("control_ranks", []))])
print(f"wrote {all_csv_path}")


def _render_pdf(html_path):
    for base in (Path.cwd(), *Path.cwd().parents):
        script = base / "src/engine/gates/notebooks/toehold/render_pdf.py"
        if script.exists():
            break
    else:
        print("render_pdf.py not found -- HTML written, PDF skipped")
        return None
    pdf_path = html_path.with_suffix(".pdf")
    subprocess.run([sys.executable, str(script), str(html_path), str(pdf_path), "--portrait"],
                   check=True)
    print(f"wrote {pdf_path}")
    return pdf_path


_pdf_made = _render_pdf(_print_html_path)   # PDF: table + one page of details per design
report_pdf_path = _pdf_made.rename(report_html_path.with_suffix(".pdf")) if _pdf_made else None
_print_html_path.unlink()   # only the PDF's source
print(f"HTML report with Details buttons -> {report_html_path}")